# Controlador PID

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, clear_output


# -------------------------------------------------------
# Vector de tiempo
# -------------------------------------------------------

t = np.linspace(0, 20, 700)


# -------------------------------------------------------
# Sliders
# -------------------------------------------------------

kp_slider = widgets.FloatSlider(
    value=1.4,
    min=0.1,
    max=10.0,
    step=0.1,
    description='kp:',
    continuous_update=False,
    layout=widgets.Layout(width='500px')
)

ki_slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=5.0,
    step=0.05,
    description='ki:',
    continuous_update=False,
    layout=widgets.Layout(width='500px')
)

kd_slider = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=5.0,
    step=0.1,
    description='kd:',
    continuous_update=False,
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()


# -------------------------------------------------------
# Función para calcular tiempo de establecimiento (2 %)
# -------------------------------------------------------

def settling_time(t, y, y_final):

    band = 0.02 * max(abs(y_final), 1e-6)

    outside = np.where(
        np.abs(y - y_final) > band
    )[0]

    if len(outside) == 0:
        return 0.0

    if outside[-1] >= len(t) - 1:
        return np.nan

    return t[outside[-1] + 1]


# -------------------------------------------------------
# Simulación PID
# -------------------------------------------------------

def simulate(change=None):

    kp = kp_slider.value
    ki = ki_slider.value
    kd = kd_slider.value


    # ---------------------------------------------------
    # Ecuación característica:
    #
    # s³ + (1.4 + kd)s² + (1 + kp)s + ki = 0
    # ---------------------------------------------------

    poles = np.roots([
        1,
        1.4 + kd,
        1 + kp,
        ki
    ])

    stable = np.all(np.real(poles) < 0)


    # ---------------------------------------------------
    # Modelo de estados
    #
    # x1 = y
    # x2 = dy/dt
    # x3 = integral del error
    #
    # Planta:
    # y'' + 1.4 y' + y = u
    #
    # Control:
    # u = kp(r-y) + ki*x3 - kd*y'
    # ---------------------------------------------------

    A = np.array([
        [0,             1,            0],
        [-(1 + kp), -(1.4 + kd),     ki],
        [-1,            0,            0]
    ])

    B = np.array([
        [0],
        [kp],
        [1]
    ])

    C = np.eye(3)

    D = np.zeros((3, 1))

    system = signal.StateSpace(A, B, C, D)


    # Referencia escalón unitario
    r = np.ones_like(t)

    tout, x, _ = signal.lsim(
        system,
        U=r,
        T=t
    )


    # ---------------------------------------------------
    # Variables
    # ---------------------------------------------------

    y = x[:, 0]
    y_dot = x[:, 1]
    integral_error = x[:, 2]

    e = r - y

    # Señal de control
    u = (
        kp * e
        + ki * integral_error
        - kd * y_dot
    )


    # ---------------------------------------------------
    # Indicadores
    # ---------------------------------------------------

    if stable:

        if ki > 0:
            y_final = 1.0
        else:
            y_final = kp / (1 + kp)

        ess = 1 - y_final

        ymax = np.max(y)

        overshoot = max(
            0,
            (ymax - y_final) / max(abs(y_final), 1e-6) * 100
        )

        ts = settling_time(
            tout,
            y,
            y_final
        )

        umax = np.max(np.abs(u))

        urms = np.sqrt(
            np.mean(u**2)
        )

    else:

        y_final = np.nan
        ess = np.nan
        overshoot = np.nan
        ts = np.nan
        umax = np.max(np.abs(u))
        urms = np.sqrt(np.mean(u**2))


    # ---------------------------------------------------
    # Mostrar resultados
    # ---------------------------------------------------

    with output:

        clear_output(wait=True)

        fig, ax = plt.subplots(
            2, 2,
            figsize=(11, 7)
        )


        # =================================================
        # 1. Respuesta
        # =================================================

        ax[0, 0].plot(
            tout,
            y,
            label='Salida y(t)'
        )

        ax[0, 0].plot(
            tout,
            r,
            linestyle='--',
            label='Referencia'
        )

        ax[0, 0].set_xlabel('Tiempo [s]')
        ax[0, 0].set_ylabel('Salida')
        ax[0, 0].set_title('Respuesta del sistema')
        ax[0, 0].grid()
        ax[0, 0].legend()


        # =================================================
        # 2. Error
        # =================================================

        ax[0, 1].plot(
            tout,
            e
        )

        ax[0, 1].axhline(
            0,
            linestyle='--'
        )

        ax[0, 1].set_xlabel('Tiempo [s]')
        ax[0, 1].set_ylabel('e(t)')
        ax[0, 1].set_title('Error de seguimiento')
        ax[0, 1].grid()


        # =================================================
        # 3. Señal de control
        # =================================================

        ax[1, 0].plot(
            tout,
            u
        )

        ax[1, 0].set_xlabel('Tiempo [s]')
        ax[1, 0].set_ylabel('u(t)')
        ax[1, 0].set_title('Señal de control')
        ax[1, 0].grid()


        # =================================================
        # 4. Polos
        # =================================================

        ax[1, 1].scatter(
            poles.real,
            poles.imag,
            marker='x',
            s=100
        )

        ax[1, 1].axhline(0)
        ax[1, 1].axvline(0)

        ax[1, 1].set_xlabel('Parte real')
        ax[1, 1].set_ylabel('Parte imaginaria')
        ax[1, 1].set_title('Polos del sistema')
        ax[1, 1].grid()

        # Límites adaptativos para no perder los polos
        max_real = max(
            2,
            np.max(np.abs(poles.real)) + 1
        )

        max_imag = max(
            2,
            np.max(np.abs(poles.imag)) + 1
        )

        ax[1, 1].set_xlim(
            -max_real,
            max(1, 0.2 * max_real)
        )

        ax[1, 1].set_ylim(
            -max_imag,
            max_imag
        )

        plt.tight_layout()
        plt.show()


        # =================================================
        # Resultados
        # =================================================

        print(f"kp = {kp:.2f}")
        print(f"ki = {ki:.2f}")
        print(f"kd = {kd:.2f}")
        print()

        if stable:

            print("Sistema ESTABLE")
            print(f"Valor final y(∞)          = {y_final:.3f}")
            print(f"Error estacionario e_ss   = {ess:.3f}")
            print(f"Sobrepaso                 = {overshoot:.1f} %")

            if np.isnan(ts):
                print("T. establecimiento        > 20 s")
            else:
                print(f"T. establecimiento        = {ts:.2f} s")

        else:

            print("Sistema INESTABLE")

        print()
        print(f"Máximo |u(t)|             = {umax:.3f}")
        print(f"RMS de u(t)               = {urms:.3f}")

        print("\nPolos:")

        for p in poles:
            print(f"   {p:.4f}")


# -------------------------------------------------------
# Actualizar cuando cambia un parámetro
# -------------------------------------------------------

kp_slider.observe(simulate, names='value')
ki_slider.observe(simulate, names='value')
kd_slider.observe(simulate, names='value')


# -------------------------------------------------------
# Mostrar controles
# -------------------------------------------------------

display(kp_slider)
display(ki_slider)
display(kd_slider)
display(output)

simulate()

FloatSlider(value=1.4, continuous_update=False, description='kp:', layout=Layout(width='500px'), max=10.0, min…

FloatSlider(value=0.5, continuous_update=False, description='ki:', layout=Layout(width='500px'), max=5.0, step…

FloatSlider(value=0.2, continuous_update=False, description='kd:', layout=Layout(width='500px'), max=5.0)

Output()